# Simple plot from high-res CESM (CAM-MPAS) simulation

#### Based on notebook by Brian Medeiros & Philip Chmielowiec, and with improvements from Organ Eroglu

In [ ]:
import uxarray as ux
import cartopy.crs as ccrs
import geoviews.feature as gf
import geoviews as gv
import holoviews as hv

from scipy.special import expit

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as mcolors
import gc

hv.extension('matplotlib')


In [ ]:
%%time

# Data files
# Note that the 'combined_10day.nc' data file I'm using below was created by extracting fields from the cam.h2a files with ncks,
# then reassembling a single file with ncrcat.  But any and all files should work, it was just easier to have one for the 
# animation / image generation loop below.
data_file = "/glade/derecho/scratch/bdobbins/ncsa_3.75km_output/combined_10day.nc"
grid_file = "/glade/campaign/cesm/cesmdata/inputdata/share/meshes/mpasa3.75_ESMFmesh-20210803.nc"

uxds = ux.open_dataset(grid_file, data_file)

In [ ]:

# Transparency over the background
alpha = 0.25

# Color bounds
vmin = 0.001
vmax = 0.7

# Backgrounds
background = gv.RGB.load_image('/glade/work/bdobbins/uxarray/world.200412.3x5400x2700.png', bounds=(-180, -90, 180, 90))
background2 = gv.RGB.load_image('/glade/work/bdobbins/uxarray/world.topo.bathy.200412.3x5400x2700.png', bounds=(-180, -90, 180, 90))

# Create a custom colormap
colors = plt.cm.Greys_r(np.linspace(0.0, 1.0, 1000))
colors[:,-1] = expit(np.arange(alpha, 1, colors.shape[0])) # sigmoid scaling of alpha

custom_cmap = plt.matplotlib.colors.ListedColormap(colors)
custom_cmap.set_over('white')
custom_cmap.set_under(alpha=0.0) # anything under vmin will be transparent
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

# Set up the projections
projection_orthographic = ccrs.Orthographic(central_longitude=-100, central_latitude=20)
projection_conus        = ccrs.PlateCarree()

xlim_globe=(-180,180)
ylim_globe=(-90,90)
xlim_conus=(-125, -66.5)
ylim_conus=(24, 50)

In [ ]:
# plot function

def makeplot(title, background, colormap, projection, xlim, ylim, global_extent, pixel_ratio, fig_size, data_in):
    data = data_in.plot.rasterize(
        method="point",
        projection=projection,
        backend="matplotlib",
        fig_size=fig_size,
        pixel_ratio=pixel_ratio,
        cmap=custom_cmap,
        norm=norm,
        clabel='condensed water path [kg m$^{-2}$]', 
        fontscale=12,
    ).opts(title=title)

    bg = background.opts(
        projection=projection,
        global_extent=global_extent,
        xlim=xlim,
        ylim=ylim
    )

    plot = bg * data
    plt.rcParams['figure.facecolor'] = 'white'
    return plot

def show_save(plot, show, save, filename):
    if (show):
        gv.output(plot)
    if (save):
        hv.save(plot, filename, fmt='png')


In [ ]:
# animation loop:

starttime = 0
endtime = 240

show = False
save = True

for i in range(starttime, endtime):
    cld = uxds['CLDTOT'].isel(time=i)
    lwp = uxds['TGCLDLWP'].isel(time=i)
    iwp = uxds['TGCLDIWP'].isel(time=i)

    cwp = lwp+iwp
    cwp_masked = cwp.where(cld>0.0)
    d = cwp_masked
    
    plot = makeplot("3.75km CESM - Condensed Water Path", background, custom_cmap, projection_orthographic, xlim_globe, ylim_globe, True,  3.0, 1600, d)
    filename="/glade/derecho/scratch/bdobbins/ncsa_plots_vmax0.7_expit_tot/ncsa_3.75km_globe_i" + str(i).rjust(5, '0')
    show_save(plot, show, save, filename)

    plot = makeplot("3.75km CESM - Condensed Water Path", background, custom_cmap, projection_conus,        xlim_conus, ylim_conus, False, 8.0, 1600, d)
    filename="/glade/derecho/scratch/bdobbins/ncsa_plots_vmax0.7_expit_tot/ncsa_3.75km_conus_i" + str(i).rjust(5, '0')
    show_save(plot, show, save, filename)

    # Cleanup
    plt.close('all')
    gc.collect()

# Running this will give you a lot of deprecation warnings on the plot.rasterize() call -- improvements are most welcome here!
